# Phase 7A — TF-IDF & Text Classification with sklearn

**Theory:** TF-IDF (Term Frequency-Inverse Document Frequency) weights words by importance.
Words that appear often in a document but rarely across all documents are the most distinctive.

**Formula:** `TF-IDF(t, d) = TF(t, d) × IDF(t)`

**Install:** `pip install scikit-learn`

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

np.random.seed(42)
sns.set_theme(style="whitegrid")

---
## 1. CountVectorizer vs TfidfVectorizer

In [ ]:
corpus = [
    "machine learning is a subset of artificial intelligence",
    "deep learning uses neural networks with many layers",
    "natural language processing handles text and language data",
    "computer vision processes images using convolutional neural networks",
    "reinforcement learning trains agents through reward and punishment",
]

# CountVectorizer — raw word counts
cv = CountVectorizer(stop_words="english")
count_matrix = cv.fit_transform(corpus).toarray()
count_df = pd.DataFrame(count_matrix, columns=cv.get_feature_names_out())
print("Count Matrix (first 10 columns):")
print(count_df.iloc[:, :10])

# TF-IDF Vectorizer
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(corpus).toarray()
tfidf_df = pd.DataFrame(tfidf_matrix, columns=tfidf.get_feature_names_out())
print("\nTF-IDF Matrix (first 10 columns):")
print(tfidf_df.iloc[:, :10].round(3))

In [ ]:
# The word 'neural' appears in docs 1 and 3 — it has lower TF-IDF than unique words
# The word 'reinforcement' appears only in doc 4 — gets higher TF-IDF score

# Top TF-IDF words per document
feature_names = tfidf.get_feature_names_out()
print("Top 3 TF-IDF words per document:")
for i, (doc, row) in enumerate(zip(corpus, tfidf_matrix)):
    top_idx = row.argsort()[-3:][::-1]
    top_words = [(feature_names[j], f"{row[j]:.3f}") for j in top_idx]
    print(f"  Doc {i}: {top_words}")

---
## 2. Text Classification with TF-IDF + Machine Learning

Build a sentiment classifier using movie reviews.

In [ ]:
# Synthetic movie review dataset
positive_reviews = [
    "This movie was absolutely fantastic and entertaining",
    "An incredible film with outstanding performances",
    "I loved every minute of this beautiful masterpiece",
    "Brilliant storytelling and amazing cinematography",
    "Wonderful movie highly recommended to everyone",
    "The acting was superb and the plot was thrilling",
    "A masterpiece of modern cinema truly remarkable",
    "Exciting and heartwarming story with great characters",
    "One of the best films I have ever seen incredible",
    "Stunning visuals and excellent direction wonderful work",
]
negative_reviews = [
    "This was the worst movie I have ever seen terrible",
    "Boring and pointless waste of time and money",
    "Horrible acting and a completely predictable plot",
    "Disappointing film with poor character development",
    "I hated this movie it was dull and uninteresting",
    "Terrible script and awful special effects bad movie",
    "A complete disaster from beginning to end just awful",
    "The worst film of the year do not waste your time",
    "Absolutely dreadful nothing works in this failure",
    "Painful to watch boring slow and completely forgettable",
]

reviews = positive_reviews + negative_reviews
labels = [1] * 10 + [0] * 10  # 1=positive, 0=negative

X_train, X_test, y_train, y_test = train_test_split(
    reviews, labels, test_size=0.3, random_state=42, stratify=labels
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

In [ ]:
# Build and compare pipelines
models = {
    "Naive Bayes + CountVec": Pipeline(
        [("cv", CountVectorizer(stop_words="english")), ("clf", MultinomialNB())]
    ),
    "Naive Bayes + TF-IDF": Pipeline(
        [("tfidf", TfidfVectorizer(stop_words="english")), ("clf", MultinomialNB())]
    ),
    "Logistic + TF-IDF": Pipeline(
        [
            ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2))),
            ("clf", LogisticRegression(max_iter=1000)),
        ]
    ),
}

print("Model Comparison:")
for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    from sklearn.metrics import accuracy_score, f1_score

    print(
        f"  {name:35s}: acc={accuracy_score(y_test, y_pred):.3f}, f1={f1_score(y_test, y_pred):.3f}"
    )

In [ ]:
# Inspect the most discriminative words
best_pipe = models["Logistic + TF-IDF"]
lr_model = best_pipe.named_steps["clf"]
tfidf_vec = best_pipe.named_steps["tfidf"]
feature_names = tfidf_vec.get_feature_names_out()

coefs = lr_model.coef_[0]
top_positive = sorted(zip(coefs, feature_names))[-10:][::-1]
top_negative = sorted(zip(coefs, feature_names))[:10]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

pos_words = [w for _, w in top_positive]
pos_coefs = [c for c, _ in top_positive]
axes[0].barh(pos_words, pos_coefs, color="green", alpha=0.7)
axes[0].set_title("Top Positive Words")
axes[0].set_xlabel("Logistic Regression Coefficient")

neg_words = [w for _, w in top_negative]
neg_coefs = [c for c, _ in top_negative]
axes[1].barh(neg_words, neg_coefs, color="red", alpha=0.7)
axes[1].set_title("Top Negative Words")

plt.suptitle("Most Discriminative Features for Sentiment", fontsize=13)
plt.tight_layout()
plt.show()

---
## Summary

| Method | Description | Advantage |
|--------|-------------|----------|
| CountVectorizer | Raw word counts | Simple, interpretable |
| TfidfVectorizer | Down-weights common words | Better for classification |
| ngram_range=(1,2) | Include bigrams ("not good") | Captures phrases |

**Best sklearn text classifiers:**
1. `MultinomialNB` — fast, great baseline for text
2. `LogisticRegression` — best for short/medium texts with TF-IDF
3. `LinearSVC` — very fast, often best with TF-IDF

**Next:** HuggingFace Transformers — learn word embeddings and BERT for much better accuracy.